In [ ]:
#python -m pip install --upgrade pip
#pip install requests
#pip install beautifulsoup4
#pip install lxml

In [1]:
import csv
import os
import time
import requests
from bs4 import BeautifulSoup

## 1. 접근 가능한 사이트인지 확인

In [2]:
url = "https://www.melon.com/chart/index.htm"
hdr = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Referer': 'https://www.melon.com/',
}

res = requests.get(url, headers=hdr)
time.sleep(1)
print(f"대상 URL: {url}, 상태코드: {res.status_code}")
print(f"수집 가능: {'🟢 허용됨' if res.status_code == 200 else '🔴 차단됨 (수집 금지)'}")

대상 URL: https://www.melon.com/chart/index.htm, 상태코드: 200
수집 가능: 🟢 허용됨


## 2. 웹 페이지에서 코드 가져오기

In [3]:
if res.status_code != 200:
    print(f"웹페이지응답코드:{res.status_code}, 웹 페이지 정보를 불러오지 못했습니다.")
else:
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "lxml")
    print(soup.title.get_text())

멜론차트>TOP100>멜론


## 3. 원하는 정보를 가지고 있는/획득할 수 있는 tag 찾기

멜론 차트는 `<div class="service_list_song">` 안의 `<table>` 에 곡 목록이 `<tr>` 로 반복됩니다.
첫 번째 tr은 제목행(th만 있음)이라, td가 있는 행만 실제 곡 정보입니다.

In [4]:
song_list_div = soup.find("div", class_="service_list_song")
all_trs = song_list_div.find_all("tr")

# 헤더 행(th만 있는 행)은 제외하고, 실제 곡 정보가 든 행(td가 있는 행)만 사용
song_rows = [tr for tr in all_trs if tr.find("td")]

nums_songs = len(song_rows)
print(f"총 곡 수:{nums_songs}")

총 곡 수:100


## 4. 곡 정보 + 앨범 이미지 추출 및 다운로드

In [5]:
img_dir_name = "멜론앨범이미지"
if not os.path.isdir(img_dir_name):
      os.mkdir(img_dir_name)
cur_dir = os.getcwd()

for tr in song_rows:
    # 순위
    rank_tag = tr.find("span", class_="rank")
    rank = rank_tag.get_text(strip=True) if rank_tag else None

    # 곡명 (rank01), 가수/앨범 (rank02)
    title_tag = tr.find("div", class_="ellipsis rank01")
    title = None
    if title_tag:
        a_tag = title_tag.find("a")
        title = a_tag.get_text(strip=True) if a_tag else title_tag.get_text(strip=True)

    artist_tag = tr.find("div", class_="ellipsis rank02")
    artist = None
    album = None
    if artist_tag:
        a_tags = artist_tag.find_all("a")
        if len(a_tags) >= 1:
            artist = a_tags[0].get_text(strip=True)
        if len(a_tags) >= 2:
            album = a_tags[1].get_text(strip=True)

    # 앨범 이미지
    img_tag = tr.find("img")
    img_url = img_tag.get("src") if img_tag else None

    print(rank, title, artist, album)

    if img_url and rank:
        safe_title = (title or "unknown").replace("/", "_")
        r = requests.get(img_url, headers=hdr)
        with open(f'{cur_dir}/{img_dir_name}/{rank}_{safe_title}.jpg', "wb") as out:
            out.write(r.content)

1 LOVE ATTACK RESCENE (리센느) RESCENE (리센느)
2 퇴사할게여 (Narr. 기안84) 소연 (SOYEON) 소연 (SOYEON)
3 갑자기 아이오아이 (I.O.I) 아이오아이 (I.O.I)
4 이 별로부터 아이유 아이유
5 BiiiG BIGBANG (빅뱅) BIGBANG (빅뱅)
6 REDRED CORTIS (코르티스) CORTIS (코르티스)
7 Pretty Girl RESCENE (리센느) RESCENE (리센느)
8 Deja Vu RESCENE (리센느) RESCENE (리센느)
9 Dear my crazy soulmate 아이유 아이유
10 BAD ATEEZ(에이티즈) ATEEZ(에이티즈)
11 만찬가 태연 (TAEYEON) 태연 (TAEYEON)
12 LEMONADE aespa aespa
13 It′s Me 아일릿(ILLIT) 아일릿(ILLIT)
14 Drowning WOODZ WOODZ
15 노스탈지아 BIG Naughty (서동현) BIG Naughty (서동현)
16 사랑하게 될 거야 한로로 한로로
17 사랑은 늘 도망가 임영웅 임영웅
18 Good Goodbye 화사 (HWASA) 화사 (HWASA)
19 RUDE! Hearts2Hearts (하츠투하츠) Hearts2Hearts (하츠투하츠)
20 Skibidi 소녀시대-효리수 (Girls' Generation-HRS) 소녀시대-효리수 (Girls' Generation-HRS)
21 캐치 캐치 YENA (최예나) YENA (최예나)
22 여름아 부탁해 볼빨간사춘기 볼빨간사춘기
23 0+0 한로로 한로로
24 Popcorn 도경수(D.O.) 도경수(D.O.)
25 생각을 멈추다 보면 최유리 최유리
26 기쁨, 슬픔, 아름다운 마음 AKMU (악뮤) AKMU (악뮤)
27 또또 임영웅 임영웅
28 어떻게 이별까지 사랑하겠어, 널 사랑하는 거지 AKMU (악뮤) AKMU (악뮤)
29 타임캡슐 다비치 다비치
30 HOME SWEET HOME (feat. 태양, 대성) G-

## 5. CSV로 저장

In [ ]:
filename = "멜론_차트.csv"
f = open(filename, "w", encoding="utf-8-sig", newline="")
writer = csv.writer(f)
attributes = ["순위", "곡명", "가수", "앨범", "이미지파일명"]
writer.writerow(attributes)

for tr in song_rows:
    rank_tag = tr.find("span", class_="rank")
    rank = rank_tag.get_text(strip=True) if rank_tag else None

    title_tag = tr.find("div", class_="ellipsis rank01")
    title = None
    if title_tag:
        a_tag = title_tag.find("a")
        title = a_tag.get_text(strip=True) if a_tag else title_tag.get_text(strip=True)

    artist_tag = tr.find("div", class_="ellipsis rank02")
    artist = None
    album = None
    if artist_tag:
        a_tags = artist_tag.find_all("a")
        if len(a_tags) >= 1:
            artist = a_tags[0].get_text(strip=True)
        if len(a_tags) >= 2:
            album = a_tags[1].get_text(strip=True)

    safe_title = (title or "unknown").replace("/", "_")
    img_filename = f"{rank}_{safe_title}.jpg"

    prod_inf = [rank, title, artist, album, img_filename]
    writer.writerow(prod_inf)

f.close()
print(f"완료: {filename} 저장됨")

In [10]:
import re
import time
import random

def extract_songid(tr):
    title_div = tr.find("div", class_="ellipsis rank01")
    if title_div is None:
        return None
    a_tag = title_div.find("a")
    if a_tag is None:
        return None
    href = a_tag.get("href", "")
    m = re.search(r"playSong\('\d+',\s*(\d+)\)", href)
    return m.group(1) if m else None

songs = []
for tr in song_rows:
    rank_tag = tr.find("span", class_="rank")
    rank = rank_tag.get_text(strip=True) if rank_tag else None

    title_tag = tr.find("div", class_="ellipsis rank01")
    title = None
    if title_tag:
        a_tag = title_tag.find("a")
        title = a_tag.get_text(strip=True) if a_tag else title_tag.get_text(strip=True)

    artist_tag = tr.find("div", class_="ellipsis rank02")
    artist = None
    if artist_tag:
        a_tags = artist_tag.find_all("a")
        if a_tags:
            artist = a_tags[0].get_text(strip=True)

    songid = extract_songid(tr)
    songs.append({"rank": rank, "title": title, "artist": artist, "songid": songid})

print(f"songid 추출된 곡 수: {sum(1 for s in songs if s['songid'])} / {len(songs)}")
print(songs[:3])

songid 추출된 곡 수: 100 / 100
[{'rank': '1', 'title': 'LOVE ATTACK', 'artist': 'RESCENE\xa0(리센느)', 'songid': '37928381'}, {'rank': '2', 'title': '퇴사할게여 (Narr. 기안84)', 'artist': '소연\xa0(SOYEON)', 'songid': '602862199'}, {'rank': '3', 'title': '갑자기', 'artist': '아이오아이\xa0(I.O.I)', 'songid': '602024048'}]


In [11]:
def get_song_detail(songid):
    detail_url = f"https://www.melon.com/song/detail.htm?songId={songid}"
    res = requests.get(detail_url, headers=hdr)
    time.sleep(random.randint(3, 5))  # 매 요청 사이 대기

    if res.status_code != 200:
        print(f"  -> 상태코드 {res.status_code}, 건너뜀")
        return None

    soup_d = BeautifulSoup(res.text, "lxml")

    genre = None
    release_date = None
    dl = soup_d.find("dl", class_="list")
    if dl:
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        info = {dt.get_text(strip=True): dd.get_text(strip=True) for dt, dd in zip(dts, dds)}
        genre = info.get("장르")
        release_date = info.get("발매일")

    lyric_tag = soup_d.find("div", id="d_video_summary")
    lyric = lyric_tag.get_text(separator="\n", strip=True) if lyric_tag else None

    return {"genre": genre, "release_date": release_date, "lyric": lyric}


# 테스트: 상위 3곡만
test_songs = songs[:3]
for s in test_songs:
    detail = get_song_detail(s["songid"])
    print(s["rank"], s["title"])
    print("  장르:", detail["genre"] if detail else None)
    print("  발매일:", detail["release_date"] if detail else None)
    print("  가사 앞부분:", (detail["lyric"][:50] + "...") if detail and detail["lyric"] else None)
    print()

1 LOVE ATTACK
  장르: 댄스
  발매일: 2024.08.27
  가사 앞부분: Feeling love attack!
I am all you need
I am all yo...

2 퇴사할게여 (Narr. 기안84)
  장르: 록/메탈
  발매일: 2026.09.07
  가사 앞부분: 퇴사할게요
아쉬울 거도 없거든요
원래 이렇게 열심히
살 생각 없었거든요
오래 했어요
수백 ...

3 갑자기
  장르: 댄스
  발매일: 2026.05.19
  가사 앞부분: 어쩌면 잘된 일이야
빨간 노을빛처럼
예쁜 널 보내야 했던 그때가
머나먼 희미한 기억
이젠 ...



In [7]:
# 첫 번째 곡의 title 링크 a 태그를 그대로 출력해서 실제 구조 확인
first_title_div = song_rows[0].find("div", class_="ellipsis rank01")
a_tag = first_title_div.find("a")
print("href:", a_tag.get("href"))
print("전체 태그:", a_tag)

href: javascript:melon.play.playSong('1000002721',37928381);
전체 태그: <a href="javascript:melon.play.playSong('1000002721',37928381);" title="LOVE ATTACK 재생">LOVE ATTACK</a>
